# Neural Network Fraud Detection Model

This notebook evaluates a feedforward neural network for mule account detection.

Unlike tree-based models, neural networks can learn complex nonlinear relationships between engineered fraud indicators and account behavior.

The objective is to determine whether deep learning can identify fraud patterns missed by gradient boosting methods.

In [ ]:
import numpy as np
import pandas as pd
df = pd.read_csv("../data/raw/DataSet.csv")

import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    average_precision_score
)

## Feature Engineering

The feature set includes both original dataset variables and engineered fraud indicators derived from exploratory analysis.

These features capture fraud clusters, high-risk behavioral regions, interaction effects, and missing-value patterns.

In [ ]:
df["F115_high"] = (df["F115"] > 0.78).astype(int)

df["F2582_hot"] = (
    (df["F2582"] > -0.03)
    & (df["F2582"] <= 0)
).astype(int)

df["F2956_hot"] = (
    (df["F2956"] > 19)
    & (df["F2956"] <= 32)
).astype(int)

df["F531_hot"] = (
    (df["F531"] > 0.95)
    & (df["F531"] <= 1.35)
).astype(int)

df["F2737_safe"] = (
    (df["F2737"] > 0)
    & (df["F2737"] <= 0.04)
).astype(int)

df["F2582_F531"] = (
    df["F2582_hot"]
    & df["F531_hot"]
).astype(int)

df["fraud_cluster_1"] = (
    df["F2582_hot"]
    & df["F2956_hot"]
    & df["F531_hot"]
).astype(int)

df["f3836_hot"] = (
    (df["F3836"] > 148.596 ) &
    (df["F3836"] <= 20077.212)
).astype(int)

df["F2956_F115"] = (
    df["F2956_hot"] &
    df["F115_high"]
).astype(int)

df["F2582_pos_F2956_low"] = (
    (df["F2582"] > 0.15) &
    (df["F2956"] < 60)
).astype(int)

In [ ]:
features = [
    "F115",
    "F2582",
    "F2956",
    "F531",
    "F2737",
    "F670",
    "F673",
    "F3891",
    "F3889",

    # engineered features

    "F115_high",
    "F2582_hot",
    "F2956_hot",
    "F531_hot",
    "F2737_safe",
    "F2582_F531",
    "fraud_cluster_1",
    "f3836_hot",
    "F2956_F115",
    "F2582_pos_F2956_low"

]

family_cols = [
    "F664","F665","F666",
    "F667","F668","F669",
    "F670","F671","F672",
    "F673","F674","F675",

    "F1","F2","F3","F4","F5","F6","F7","F8","F9","F10","F12"
]

for col in family_cols:
    df[f"{col}_missing"] = df[col].isna().astype(int)

features += [f"{col}_missing" for col in family_cols]

In [ ]:
X = pd.get_dummies(
    df[features],
    columns=["F3891", "F3889"],
    dummy_na=True
)

y = df["F3924"]

## Data Preparation

Categorical variables are encoded using one-hot encoding and the dataset is split using stratified sampling.

The fraud prevalence is preserved across training and testing datasets.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

X_train = X_train.fillna(-999)
X_test = X_test.fillna(-999)

In [ ]:
X_train_nn = X_train.copy()
X_test_nn = X_test.copy()

## Missing Value Handling

Neural networks cannot directly process missing values.

Missing observations are therefore replaced with a constant placeholder value prior to scaling.

In [ ]:
X_train_nn = X_train_nn.fillna(-999)
X_test_nn = X_test_nn.fillna(-999)

## Feature Scaling

Neural networks are sensitive to feature magnitudes.

All numerical inputs are standardized using StandardScaler to improve optimization stability and convergence.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_nn)
X_test_scaled = scaler.transform(X_test_nn)

## Class Imbalance Handling

Fraudulent accounts represent a very small fraction of the dataset.

Class weights are computed and applied during training to increase the penalty associated with misclassifying fraud cases.

In [ ]:
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=y_train
)

class_weights = {
    0: weights[0],
    1: weights[1]
}

print(class_weights)

## Network Architecture

A fully connected feedforward neural network is trained using ReLU activations and dropout regularization.

The output layer uses a sigmoid activation to produce fraud probabilities.

In [ ]:
model = tf.keras.Sequential([

    tf.keras.layers.Dense(
        64,
        activation="relu",
        input_shape=(X_train_scaled.shape[1],)
    ),

    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(
        32,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(
        1,
        activation="sigmoid"
    )
])

## Model Training

The network is trained using binary cross-entropy loss and the Adam optimizer.

Model performance is monitored using AUC during training.

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["AUC"]
)

In [ ]:
history = model.fit(
    X_train_scaled,
    y_train,

    epochs=50,

    batch_size=64,

    validation_split=0.2,

    class_weight=class_weights,

    verbose=1
)

In [ ]:
y_prob_nn = model.predict(X_test_scaled).flatten()

## Model Evaluation

Performance is evaluated across multiple probability thresholds to understand the precision-recall tradeoff.

Because fraud detection is highly imbalanced, recall is treated as a primary evaluation metric.

In [ ]:
for threshold in [0.5, 0.4, 0.3, 0.25, 0.2]:

    y_pred = (y_prob_nn >= threshold).astype(int)

    print(f"\nThreshold: {threshold}")

    print(confusion_matrix(y_test, y_pred))

    print(classification_report(y_test, y_pred))

## Prediction Export

Predicted probabilities are exported for comparison with gradient boosting models and subsequent ensemble construction.

In [ ]:
nn_probs = pd.DataFrame({
    "prob": y_prob_nn.flatten()
}, index=X_test.index)

nn_probs.to_csv("../data/processed/nn_probs.csv")

In [ ]:
ap = average_precision_score(
    y_test,
    y_prob_nn
)

print("Average Precision:", ap)

In [ ]:
results = pd.DataFrame({
    "actual": y_test,
    "prob": y_prob_nn
}, index=X_test.index)

frauds = results[
    results["actual"] == 1
]

frauds.sort_values("prob")

## Training Dynamics

The learning curve below illustrates model performance throughout training. Monitoring both training and validation AUC helps identify convergence behaviour and potential overfitting.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.plot(history.history["AUC"], label="Train AUC")
plt.plot(history.history["val_AUC"], label="Validation AUC")

plt.title("Neural Network Training")
plt.xlabel("Epoch")
plt.ylabel("AUC")
plt.legend()

plt.show()

plt.savefig(
    "../figures/nn_training_curve.png",
    bbox_inches="tight"
)

## Key Findings

The neural network achieved the strongest fraud recall among all evaluated models.

Key observations:

- Nonlinear relationships between engineered features contributed substantially to performance.
- Class weighting successfully improved fraud sensitivity.
- The model identified fraud cases missed by multiple tree-based approaches.
- The resulting probabilities were later incorporated into ensemble experiments.